# 01. 100m 격자 최종 취약지수

$$F_i=0.5P_i+0.5N_i$$

- 물리적 $P_i$: 제공 파일의 `overall_accessibility_deficit_minmax_0_1`(β=3). 높을수록 접근성이 나쁘다.
- 비물리적 $N_i$: 기존 `비물리적지수`. 두 값을 재정규화하지 않는다.
- 따라서 $F_i=0.5P_i+0.25D_i+0.25C_i$이다.
- 실제 물리적 파일은 시설분류 `전체`만 포함한다. 분야별 자료를 추가로 만들어 합산하지 않는다.
- 대상인구가 있는 격자 전체를 계산한다. 인구 0 격자에 물리적 점수를 임의 대입하지 않는다.
- 기존 방식 B의 접근성 선별 기준은 미정이다. 아래 `PHYSICAL_THRESHOLD=None`은 **후보 미선정**을 뜻한다.
  확정된 0~1 부족도 기준을 입력하면 그 이상인 격자의 후보 순위를 계산한다. 점수 자체는 재정규화하지 않는다.
- 전체 비교 순위와 실제 후보 순위를 구분한다. 대리 적용 제외 점수는 검토용이다.
- 기존 입력 파일은 읽기 전용으로 사용한다. 결과만 `data/processed/final_score`에 저장한다.

물리적 부족도는 제공된 전체 대상 격자에서 $P_i=1-(A_i-\min A)/(\max A-\min A)$로 계산된 값이다. 여기서 $A_i$는 β=3 시설 접근성 합계다. 파일에 저장된 정규화 결과를 검증하여 그대로 사용하며, 후보 선별 후 다시 정규화하지 않는다.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
start=Path.cwd().resolve()
ROOT=next(p for p in [start,*start.parents] if (p/'data/raw').is_dir() and (p/'code').is_dir())
OUT=ROOT/'data/processed/final_score/서울시_100m_최종취약지수'
pd.set_option('display.max_columns',30)

In [2]:
PHYSICAL_THRESHOLD=None  # 예시값을 자동 적용하지 않는다. 정책 기준 확정 후 0~1 값 입력
if PHYSICAL_THRESHOLD is not None:
    assert np.isfinite(PHYSICAL_THRESHOLD) and 0<=PHYSICAL_THRESHOLD<=1
physical=pd.read_csv(ROOT/'data/processed/physical_index/grid_category_accessibility_beta_sensitivity.csv',dtype={'GRID_CD':str,'행정동코드':str})
nonphysical=pd.read_csv(ROOT/'data/processed/nonphysical_index/서울시_100m_비물리적지수.csv',dtype={'GRID_CD':str,'행정동코드':str})
target=nonphysical.loc[nonphysical['취약노인수'].gt(0)].copy()
pcol='overall_accessibility_deficit_minmax_0_1'
assert not physical['GRID_CD'].duplicated().any(), '물리적 파일에 분야별 중복이 생겼는지 확인'
assert not nonphysical['GRID_CD'].duplicated().any()
assert set(physical['facility_category'])=={'전체'}
assert set(physical['GRID_CD'])==set(target['GRID_CD']), '대상 격자 집합 불일치: 연결 전 원인 확인 필요'
assert physical['overall_facility_count'].gt(0).all()
access=physical['overall_accessibility_beta_3_sum']
assert access.notna().all() and np.isfinite(access).all()
access_range=access.max()-access.min()
assert access_range>0, '접근성 값이 모두 같으면 최소–최대 정규화 기준 재검토 필요'
expected=(access-access.min())/access_range
assert np.allclose(physical['overall_accessibility_beta_3_minmax_0_1'],expected)
assert np.allclose(physical['overall_accessibility_beta_3_minmax_0_100'],100*expected)
assert np.allclose(physical[pcol],1-physical['overall_accessibility_beta_3_minmax_0_1'])
assert physical[pcol].notna().all() and physical[pcol].between(0,1).all()
assert target['비물리적지수'].notna().all() and target['비물리적지수'].between(0,1).all()
assert np.isfinite(physical[pcol]).all() and np.isfinite(target['비물리적지수']).all()
display(physical[[pcol,'overall_accessibility_beta_3_minmax_0_1']].describe())

,overall_accessibility_deficit_minmax_0_1,overall_accessibility_beta_3_minmax_0_1
count,21263.000000,21263.000000
mean,0.505263,0.494737
std,0.202428,0.202428
min,0.000000,0.000000
25%,0.351232,0.361281
50%,0.462282,0.537718
75%,0.638719,0.648768
max,1.000000,1.000000


## 1. 일대일 연결과 인구·행정동 대조

In [3]:
physical_cols=['GRID_CD','행정동코드','시군구','행정동','취약노인수',pcol,
 'overall_accessibility_beta_3_sum','overall_facility_count','facility_count_any_feasible_mode']
joined=target.merge(physical[physical_cols],on='GRID_CD',how='left',validate='one_to_one',suffixes=('','_물리'))
for col in ['행정동코드','시군구','행정동','취약노인수']:
    assert joined[col].eq(joined[col+'_물리']).all(), col+' 불일치'
result=joined.drop(columns=[c+'_물리' for c in ['행정동코드','시군구','행정동','취약노인수']]).copy()
result=result.rename(columns={pcol:'물리적부족도'})
assert len(result)==len(target) and result['취약노인수'].sum()==target['취약노인수'].sum()

## 2. 50:50 결합과 후보 설정

In [4]:
result['물리적기여점수']=.5*result['물리적부족도']
result['비물리적기여점수']=.5*result['비물리적지수']
result['최종취약지수']=result['물리적기여점수']+result['비물리적기여점수']
result['최종취약순위_전체']=result['최종취약지수'].rank(ascending=False,method='average')
result['최종취약지수_대리적용제외']=result['최종취약지수'].mask(result['경계대리적용'])
result['접근성후보']=pd.Series(pd.NA,index=result.index,dtype='boolean')
result['최종취약순위_후보']=np.nan
result['후보선별기준']=PHYSICAL_THRESHOLD if PHYSICAL_THRESHOLD is not None else np.nan
if PHYSICAL_THRESHOLD is not None:
    mask=result['물리적부족도'].ge(PHYSICAL_THRESHOLD)
    result['접근성후보']=mask
    result.loc[mask,'최종취약순위_후보']=result.loc[mask,'최종취약지수'].rank(ascending=False,method='average')
    print('선별 기준:',PHYSICAL_THRESHOLD,'후보 격자:',mask.sum())
else:
    print('접근성 후보 기준 미정: 전체 비교 점수·순위만 산정. 후보 여부·후보 순위는 결측 유지.')

접근성 후보 기준 미정: 전체 비교 점수·순위만 산정. 후보 여부·후보 순위는 결측 유지.


## 3. 검증 및 결과 저장

In [5]:
checks={
 '격자 중복 없음':not result['GRID_CD'].duplicated().any(),
 '대상 격자 완전 연결':set(result['GRID_CD'])==set(target['GRID_CD']),
 '인구 총량 보존':result['취약노인수'].sum()==target['취약노인수'].sum(),
 '최종 점수 유한값':np.isfinite(result['최종취약지수']).all(),
 '0~1 범위':result['최종취약지수'].between(0,1).all(),
 '50:50 산정식 일치':np.allclose(result['최종취약지수'],.5*result['물리적부족도']+.5*result['비물리적지수']),
 '원 구성요소 산정식 일치':np.allclose(result['최종취약지수'],.5*result['물리적부족도']+.25*result['인구규모점수']+.25*result['커뮤니케이션부족도']),
 '대리 적용 결측 일치':result['최종취약지수_대리적용제외'].isna().equals(result['경계대리적용']),
 '순위 일치':np.allclose(result['최종취약순위_전체'],result['최종취약지수'].rank(ascending=False,method='average')),
}
display(pd.Series(checks,name='통과').to_frame())
assert all(checks.values())
OUT.mkdir(parents=True,exist_ok=True)
path=OUT/'서울시_100m_최종취약지수.csv'
result.to_csv(path,index=False,encoding='utf-8-sig')
saved=pd.read_csv(path,dtype={'GRID_CD':str,'행정동코드':str})
assert len(saved)==len(result) and not saved['GRID_CD'].duplicated().any()
assert saved['취약노인수'].sum()==target['취약노인수'].sum()
assert np.allclose(saved['최종취약지수'],result['최종취약지수'])
assert saved['접근성후보'].isna().equals(result['접근성후보'].isna())
display(result[['물리적부족도','비물리적지수','최종취약지수']].describe())
print('저장:',path.relative_to(ROOT),'격자:',len(saved),'취약노인:',saved['취약노인수'].sum())

,통과
격자 중복 없음,True
대상 격자 완전 연결,True
인구 총량 보존,True
최종 점수 유한값,True
0~1 범위,True
50:50 산정식 일치,True
원 구성요소 산정식 일치,True
대리 적용 결측 일치,True
순위 일치,True


,물리적부족도,비물리적지수,최종취약지수
count,21263.000000,21263.000000,21263.000000
mean,0.505263,0.507165,0.506214
std,0.202428,0.180036,0.136927
min,0.000000,0.070621,0.136803
25%,0.351232,0.356889,0.403325
50%,0.462282,0.507764,0.502852
75%,0.638719,0.655929,0.601123
max,1.000000,0.893086,0.941066


저장: data\processed\final_score\서울시_100m_최종취약지수\서울시_100m_최종취약지수.csv 격자: 21263 취약노인: 168748
